# Лабораторная работа №6
## Анимация детерминированной динамики SIR-модели

В данном скрипте строится GIF-анимация динамики SIR-модели,
реализованной в подходе сетей Петри.

В отличие от обычного графика, анимация позволяет увидеть,
как во времени изменяется распределение популяции между состояниями:

- `S` — восприимчивые;
- `I` — инфицированные;
- `R` — выздоровевшие.

Каждый кадр анимации соответствует отдельному моменту времени,
а высота столбцов показывает численность соответствующей группы.

In [ ]:
using DrWatson
@quickactivate "project"

## Подключение модели

Основная реализация SIR-модели вынесена в файл `src/SIRPetri.jl`.
В нём определены функции построения сети Петри, выполнения симуляции
и построения графиков.

In [ ]:
include(srcdir("SIRPetri.jl"))
using .SIRPetri

using DataFrames, CSV, Plots

## Параметры модели

Для построения анимации используются базовые параметры модели:

- `β = 0.3` — коэффициент заражения;
- `γ = 0.1` — коэффициент выздоровления;
- `tmax = 100.0` — максимальное время моделирования.

Эти параметры соответствуют базовому сценарию лабораторной работы.

In [ ]:
β = 0.3
γ = 0.1
tmax = 100.0

## Построение сети Петри

Функция `build_sir_network` создаёт сеть Петри для модели SIR.

В сети используются три позиции:

- `S` — восприимчивые;
- `I` — инфицированные;
- `R` — выздоровевшие.

Переход `infection` описывает заражение, а переход `recovery`
описывает выздоровление.

In [ ]:
net, u0, states = build_sir_network(β, γ)

## Детерминированная симуляция

Для анимации используется детерминированная симуляция.
Она даёт плавную динамику изменения численности групп `S`, `I`, `R`
во времени.

Параметр `saveat = 0.2` задаёт шаг сохранения результата.
Чем меньше этот шаг, тем больше кадров будет в анимации.

In [ ]:
df = simulate_deterministic(
    net,
    u0,
    (0.0, tmax),
    saveat = 0.2,
    rates = [β, γ],
)

## Построение анимации

На каждом кадре строится столбчатая диаграмма.

По оси X расположены состояния модели:

- `S`;
- `I`;
- `R`.

По оси Y показана численность соответствующей группы.
Заголовок каждого кадра содержит текущее время моделирования.

In [ ]:
anim = @animate for i in 1:nrow(df)
    values = [df.S[i], df.I[i], df.R[i]]

    bar(
        string.(states),
        values,
        ylim = (0, 1000),
        xlabel = "State",
        ylabel = "Population",
        title = "SIR dynamics, t = $(round(df.time[i], digits = 1))",
        legend = false,
    )
end

## Сохранение результата

Полученная анимация сохраняется в файл `plots/sir_animation.gif`.

Параметр `fps = 20` задаёт скорость воспроизведения анимации.

In [ ]:
gif(anim, plotsdir("sir_animation.gif"), fps = 20)

## Итог

После выполнения скрипта в каталоге `plots/` появляется файл:

- `sir_animation.gif`.

Эта анимация наглядно показывает, как в процессе моделирования
уменьшается число восприимчивых, изменяется число инфицированных
и увеличивается число выздоровевших.

In [ ]:
println("Анимация сохранена в plots/sir_animation.gif")